# BOLDGenotyper Plot Customization Tutorial

This notebook demonstrates how to customize plots using the `plot_config.yaml` file and regenerate them with Python scripts.

## Table of Contents

1. [Understanding plot_config.yaml](#understanding-plot_config-yaml)
2. [Common Errors and Solutions](#common-errors-and-solutions)
3. [Example Customizations](#example-customizations)
4. [Complete Configuration Reference](#complete-configuration-reference)
5. [Regenerating Plots](#regenerating-plots)

## Understanding plot_config.yaml

The `plot_config.yaml` file controls all aspects of plot appearance. It's located in your output directory:

```
{organism}_output/plots/plot_config.yaml
```

### Basic Structure

```yaml
general:          # Overall settings (size, resolution, formats)
colors:           # Color mapping for each genotype
filters:          # Include/exclude specific genotypes
map:              # Map-specific settings (projection, colors)
bars:             # Bar chart settings (orientation, width)
identity:         # Identity histogram settings (bins, overlays)
```

## Common Errors and Solutions

### Error 1: "only list-like objects are allowed to be passed to isin(), you passed a `str`"

**Cause**: YAML interprets single values without brackets as strings, not lists.

**❌ Wrong**:
```yaml
filters:
  include_genotypes: "Consensus_1_S._lewini"  # String, not list!
  exclude_genotypes: "Consensus_10_S._lewini"
```

**✅ Correct**:
```yaml
filters:
  include_genotypes: ["Consensus_1_S._lewini"]  # List with one item
  exclude_genotypes: ["Consensus_10_S._lewini"]
```

**✅ Also Correct** (multi-line list):
```yaml
filters:
  include_genotypes:
    - "Consensus_1_S._lewini"
    - "Consensus_2_S._lewini"
  exclude_genotypes:
    - "Consensus_10_S._lewini"
```

**✅ Empty list** (no filtering):
```yaml
filters:
  include_genotypes: []  # Include all
  exclude_genotypes: []  # Exclude none
```

### Error 2: "'consensus_group_sp'"

**Cause**: Your data doesn't have the `consensus_group_sp` column (species-labeled genotypes).

**Solution**: This is now handled automatically! The code falls back to `consensus_group` if `consensus_group_sp` doesn't exist. No action needed.

If you still get this error, check that your CSV files have at least one of these columns:
- `consensus_group_sp` (preferred)
- `consensus_group` (fallback)

### Error 3: Color values not applying

**Cause**: Color keys must exactly match genotype names in your data.

**❌ Wrong**:
```yaml
colors:
  Consensus_1: "#E41A1C"  # Won't work if genotypes use species labels
```

**✅ Correct**:
```yaml
colors:
  Consensus_1_S._lewini: "#E41A1C"  # Must match exact genotype name
```

**Tip**: Check your genotype names in the CSV files:
```bash
# In your plots/ directory
cut -d',' -f2 data/genotype_colors.csv | tail -n +2 | sort -u
```

## Example Customizations

### Example 1: High-Resolution Publication Figures

In [ ]:
# Save this as your plot_config.yaml

publication_config = """
general:
  output_format: ['pdf', 'png', 'svg']  # Multiple formats
  dpi: 600                               # High resolution
  width_inches: 14                       # Larger figures
  height_inches: 10

colors:
  Consensus_1_S._lewini: "#E41A1C"      # Red
  Consensus_2_S._lewini: "#377EB8"      # Blue
  Consensus_3_S._lewini: "#4DAF4A"      # Green
  Consensus_4_S._lewini: "#984EA3"      # Purple
  Consensus_5_S._lewini: "#FF7F00"      # Orange

filters:
  include_genotypes: []                  # Include all genotypes
  exclude_genotypes: []                  # Don't exclude any

map:
  projection: "robinson"                 # Best for global view
  center_longitude: 0                    # Center on Atlantic
  show_country_borders: true
  border_color: "gray70"
  border_width: 0.3
  ocean_color: "#E8F4F8"                # Light blue
  land_color: "#F5F5F5"                 # Light gray
  point_alpha: 0.8                       # Slightly transparent
  point_size_range: [3, 12]             # Min and max point sizes
  point_stroke: 0.5                      # Black outline
  legend_position: "right"
  legend_title: "Genotype"

bars:
  orientation: "vertical"                # Standard bar chart
  bar_width: 0.8
  axis_text_angle: 45                    # Angled labels
  axis_text_size: 11

identity:
  binwidth: 0.5                          # Half-percent bins
  show_mean: true
  show_median: true
  show_density: false                    # Clean histogram
  stat_line_color: "red"
  stat_line_type: "dashed"
  x_limits: [95, 100]                    # Focus on high identity
  x_breaks: [95, 96, 97, 98, 99, 100]
"""

# Write to file
with open('plot_config_publication.yaml', 'w') as f:
    f.write(publication_config)
    
print("✓ Saved publication configuration")

### Example 2: Focus on Pacific Ocean

In [ ]:
pacific_config = """
general:
  output_format: ['pdf', 'png']
  dpi: 300
  width_inches: 12
  height_inches: 9

colors:
  Consensus_1_S._lewini: "#d73027"  # Darker red
  Consensus_2_S._lewini: "#4575b4"  # Darker blue
  Consensus_3_S._lewini: "#91cf60"  # Brighter green

filters:
  include_genotypes: []  # Include all
  exclude_genotypes: []  # Exclude none

map:
  projection: "mollweide"      # Good for Pacific-centered
  center_longitude: -180       # Center on Pacific Ocean
  show_country_borders: true
  border_color: "gray60"
  border_width: 0.4
  ocean_color: "#c6dbef"       # Medium blue
  land_color: "#f0f0f0"        # Light gray
  point_alpha: 0.9             # More opaque
  point_size_range: [4, 15]    # Larger points
  point_stroke: 0.8            # Thicker outline
  legend_position: "right"
  legend_title: "COI Genotype"

bars:
  orientation: "vertical"
  bar_width: 0.75
  axis_text_angle: 45
  axis_text_size: 10

identity:
  binwidth: 0.3
  show_mean: true
  show_median: false
  show_density: true
  density_alpha: 0.3
  stat_line_color: "darkred"
  x_limits: [96, 100]
  x_breaks: [96, 97, 98, 99, 100]
"""

with open('plot_config_pacific.yaml', 'w') as f:
    f.write(pacific_config)
    
print("✓ Saved Pacific-focused configuration")

### Example 3: Horizontal Bar Charts (Space-Saving)

In [ ]:
horizontal_config = """
general:
  output_format: ['pdf']
  dpi: 400
  width_inches: 10
  height_inches: 12  # Taller for horizontal bars

colors:
  Consensus_1_S._lewini: "#8dd3c7"
  Consensus_2_S._lewini: "#ffffb3"
  Consensus_3_S._lewini: "#bebada"
  Consensus_4_S._lewini: "#fb8072"
  Consensus_5_S._lewini: "#80b1d3"

filters:
  include_genotypes: []  # All genotypes
  exclude_genotypes: []  # No exclusions

map:
  projection: "platecarree"  # Simple rectangular
  center_longitude: 0
  show_country_borders: true
  border_color: "gray70"
  border_width: 0.3
  ocean_color: "#E8F4F8"
  land_color: "#F5F5F5"
  point_alpha: 0.7
  point_size_range: [2, 10]
  point_stroke: 0.5
  legend_position: "right"

bars:
  orientation: "horizontal"  # Key change: horizontal bars!
  bar_width: 0.8
  axis_text_angle: 0         # No angle needed for horizontal
  axis_text_size: 11

identity:
  binwidth: 0.5
  show_mean: true
  show_median: true
  show_density: false
  x_limits: [95, 100]
"""

with open('plot_config_horizontal.yaml', 'w') as f:
    f.write(horizontal_config)
    
print("✓ Saved horizontal bar chart configuration")

### Example 4: Filter to Top 3 Genotypes Only

In [ ]:
filtered_config = """
general:
  output_format: ['pdf', 'png']
  dpi: 300
  width_inches: 10
  height_inches: 8

colors:
  Consensus_1_S._lewini: "#e41a1c"
  Consensus_2_S._lewini: "#377eb8"
  Consensus_3_S._lewini: "#4daf4a"

filters:
  # IMPORTANT: Use list syntax with brackets!
  include_genotypes:
    - "Consensus_1_S._lewini"
    - "Consensus_2_S._lewini" 
    - "Consensus_3_S._lewini"
  exclude_genotypes: []  # Not needed when using include

map:
  projection: "robinson"
  center_longitude: 0
  show_country_borders: true
  border_color: "gray70"
  border_width: 0.3
  ocean_color: "#E8F4F8"
  land_color: "#F5F5F5"
  point_alpha: 0.8
  point_size_range: [3, 12]
  point_stroke: 0.5
  legend_position: "right"
  legend_title: "Top 3 Genotypes"

bars:
  orientation: "vertical"
  bar_width: 0.8
  axis_text_angle: 45
  axis_text_size: 10

identity:
  binwidth: 0.5
  show_mean: true
  show_median: false
  show_density: true
  density_alpha: 0.3
  x_limits: [95, 100]
"""

with open('plot_config_filtered.yaml', 'w') as f:
    f.write(filtered_config)
    
print("✓ Saved filtered (top 3) configuration")

### Example 5: Exclude Rare Genotypes

In [ ]:
exclude_rare_config = """
general:
  output_format: ['pdf']
  dpi: 300
  width_inches: 10
  height_inches: 8

colors:
  Consensus_1_S._lewini: "#1b9e77"
  Consensus_2_S._lewini: "#d95f02"
  Consensus_3_S._lewini: "#7570b3"

filters:
  include_genotypes: []  # Include all by default
  # Exclude rare genotypes (those with few samples)
  exclude_genotypes:
    - "Consensus_10_S._lewini"  # Example: rare genotype
    - "Consensus_11_S._lewini"  # Example: rare genotype

map:
  projection: "robinson"
  center_longitude: 0
  show_country_borders: true
  border_color: "gray70"
  border_width: 0.3
  ocean_color: "#E8F4F8"
  land_color: "#F5F5F5"
  point_alpha: 0.8
  point_size_range: [3, 12]
  point_stroke: 0.5
  legend_position: "right"
  legend_title: "Major Genotypes"

bars:
  orientation: "vertical"
  bar_width: 0.8
  axis_text_angle: 45
  axis_text_size: 10

identity:
  binwidth: 0.5
  show_mean: true
  show_median: true
  x_limits: [95, 100]
"""

with open('plot_config_exclude_rare.yaml', 'w') as f:
    f.write(exclude_rare_config)
    
print("✓ Saved configuration excluding rare genotypes")

## Complete Configuration Reference

### General Settings

In [ ]:
general_settings = """
general:
  output_format: ['pdf', 'png', 'svg']  # Formats to generate
  dpi: 300                              # Resolution (72-600)
  width_inches: 10                      # Figure width
  height_inches: 8                      # Figure height

# Common DPI values:
# - 72: Screen viewing
# - 150: Draft quality
# - 300: Standard publication
# - 600: High-quality publication

# Common sizes (inches):
# - Single column: 3.5 x 3.5
# - 1.5 column: 5.5 x 4.5
# - Double column: 7.0 x 5.5
# - Full page: 8.5 x 11
"""

print(general_settings)

### Color Settings

In [ ]:
color_settings = """
colors:
  # Format: "GenotypeName": "#HEXCOLOR"
  # Genotype names must EXACTLY match those in your data
  
  Consensus_1_S._lewini: "#E41A1C"  # Red
  Consensus_2_S._lewini: "#377EB8"  # Blue
  Consensus_3_S._lewini: "#4DAF4A"  # Green
  
  # If genotype not specified, automatic color assigned
  # Colors should be distinct and colorblind-friendly

# Useful color palettes:
# ColorBrewer Set1: #E41A1C, #377EB8, #4DAF4A, #984EA3, #FF7F00
# ColorBrewer Dark2: #1B9E77, #D95F02, #7570B3, #E7298A, #66A61E
# Viridis-like: #440154, #31688E, #35B779, #FDE724

# Color picker tools:
# - https://colorbrewer2.org/ (scientific palettes)
# - https://coolors.co/ (palette generator)
"""

print(color_settings)

### Filter Settings

In [ ]:
filter_settings = """
filters:
  # Include only specific genotypes
  include_genotypes: []
  # Options:
  # - [] (empty): Include all genotypes
  # - ["Gen1", "Gen2"]: Include only these
  # - ["Gen1"]: Single genotype (must use brackets!)
  
  # Exclude specific genotypes
  exclude_genotypes: []
  # Options:
  # - [] (empty): Exclude none
  # - ["Gen10", "Gen11"]: Exclude these
  # - ["Gen10"]: Single genotype (must use brackets!)
  
  # Note: If include_genotypes is not empty, exclude_genotypes is ignored

# Examples:

# Show all genotypes:
filters:
  include_genotypes: []
  exclude_genotypes: []

# Show only top 3:
filters:
  include_genotypes:
    - "Consensus_1_S._lewini"
    - "Consensus_2_S._lewini"
    - "Consensus_3_S._lewini"
  exclude_genotypes: []

# Exclude rare genotypes:
filters:
  include_genotypes: []
  exclude_genotypes:
    - "Consensus_10_S._lewini"
    - "Consensus_11_S._lewini"
"""

print(filter_settings)

### Map Settings

In [ ]:
map_settings = """
map:
  # Map projection
  projection: "robinson"
  # Options: robinson, mollweide, mercator, platecarree
  # - robinson: Balanced, good for global (default)
  # - mollweide: Oval, equal-area, good for Pacific-centered
  # - mercator: Rectangular, familiar but distorts poles
  # - platecarree: Simple rectangular (lat/lon)
  
  # Center longitude (degrees)
  center_longitude: 0
  # 0: Atlantic center, -180: Pacific center
  
  # Border settings
  show_country_borders: true
  border_color: "gray70"  # Color name or hex
  border_width: 0.3       # Line width
  
  # Base colors
  ocean_color: "#E8F4F8"  # Light blue
  land_color: "#F5F5F5"   # Light gray
  
  # Point styling
  point_alpha: 0.7        # Transparency (0-1)
  point_size_range: [2, 10]  # [min, max] sizes
  point_stroke: 0.5       # Outline width (0 = no outline)
  
  # Legend
  legend_position: "right"  # right, left, top, bottom
  legend_title: "Genotype"
"""

print(map_settings)

### Bar Chart Settings

In [ ]:
bar_settings = """
bars:
  # Orientation
  orientation: "vertical"  # vertical or horizontal
  
  # Bar width (0-1)
  bar_width: 0.8
  # 0.7: Narrow bars with gaps
  # 0.8: Standard (default)
  # 0.95: Wide bars, minimal gaps
  
  # Axis label styling
  axis_text_angle: 45  # Label rotation (0-90)
  axis_text_size: 10   # Font size
  
  # For horizontal bars:
  # - orientation: "horizontal"
  # - axis_text_angle: 0 (no rotation needed)
"""

print(bar_settings)

### Identity Histogram Settings

In [ ]:
identity_settings = """
identity:
  # Bin width (percent)
  binwidth: 0.5
  # 0.3: Fine bins, detailed
  # 0.5: Standard (default)
  # 1.0: Coarse bins, simplified
  
  # Statistical overlays
  show_mean: true     # Vertical line at mean
  show_median: true   # Vertical line at median
  show_density: false # Density curve overlay
  
  # Density curve (if show_density: true)
  density_alpha: 0.3  # Transparency (0-1)
  
  # Statistical line styling
  stat_line_color: "red"      # Color
  stat_line_type: "dashed"    # dashed, dotted, solid
  
  # Axis limits and breaks
  x_limits: [95, 100]         # [min, max] or null for auto
  x_breaks: [95, 96, 97, 98, 99, 100]  # Tick marks or null
  
  # Example: Focus on high identity (>97%)
  # x_limits: [97, 100]
  # x_breaks: [97, 98, 99, 100]
"""

print(identity_settings)

## Regenerating Plots

After editing `plot_config.yaml`, regenerate plots using the Python scripts.

### Method 1: From Command Line

In [ ]:
%%bash
# Navigate to your plots directory
cd /path/to/your_output/plots

# Regenerate all plots
python scripts/regenerate_all.py

# Or regenerate individual plots
python scripts/regenerate_map.py
python scripts/regenerate_bars.py
python scripts/regenerate_identity.py

### Method 2: From Python

In [ ]:
from pathlib import Path
from boldgenotyper.plot_regeneration import regenerate_all_plots

# Path to your plots directory
plots_dir = Path("/path/to/your_output/plots")

# Regenerate all plots
results = regenerate_all_plots(plots_dir)

# Check what was generated
for plot_type, files in results.items():
    print(f"{plot_type}: {len(files)} files")
    for f in files:
        print(f"  - {f}")

### Method 3: Individual Plot Functions

In [ ]:
from pathlib import Path
from boldgenotyper.plot_regeneration import (
    regenerate_distribution_map,
    regenerate_bar_charts,
    regenerate_identity_distribution
)

plots_dir = Path("/path/to/your_output/plots")

# Regenerate only the map
map_files = regenerate_distribution_map(plots_dir)
print(f"Generated {len(map_files)} map files")

# Regenerate only bar charts
bar_files = regenerate_bar_charts(plots_dir)
print(f"Generated {len(bar_files)} bar chart files")

# Regenerate only identity histogram
id_files = regenerate_identity_distribution(plots_dir)
print(f"Generated {len(id_files)} identity histogram files")

## Troubleshooting

### Problem: Plots look the same after regeneration

**Solution**: Check that:
1. You saved `plot_config.yaml` after editing
2. You're regenerating in the correct directory
3. Custom plots have `_custom` suffix (check `../visualization/`)

### Problem: "ModuleNotFoundError: No module named 'boldgenotyper'"

**Solution**:

In [ ]:
# Make sure boldgenotyper environment is activated
# In terminal:
# conda activate boldgenotyper

# Check installation:
import sys
print(sys.prefix)  # Should show boldgenotyper environment

# If not installed, install it:
# pip install -e /path/to/boldgenotyper/repo

### Problem: "FileNotFoundError: plot_config.yaml not found"

**Solution**: Make sure you're running scripts from the `plots/` directory:

In [ ]:
import os
from pathlib import Path

# Check current directory
print(f"Current directory: {os.getcwd()}")

# Should end with "plots"
# If not, navigate there:
os.chdir("/path/to/your_output/plots")

# Verify plot_config.yaml exists
config_path = Path("plot_config.yaml")
print(f"Config exists: {config_path.exists()}")

## Quick Reference Card

In [ ]:
quick_reference = """
╔════════════════════════════════════════════════════════════════╗
║              YAML SYNTAX QUICK REFERENCE                       ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  ✓ CORRECT                    ✗ WRONG                         ║
║  ────────────────────────────────────────────────────────      ║
║  Single value:                                                 ║
║  include: ["Gen1"]            include: "Gen1"                 ║
║                                                                ║
║  Multiple values:                                              ║
║  include:                     include: [Gen1, Gen2]           ║
║    - "Gen1"                   (missing quotes)                ║
║    - "Gen2"                                                    ║
║                                                                ║
║  Empty list:                                                   ║
║  include: []                  include:                        ║
║                               (not defined)                    ║
║                                                                ║
║  Comments:                                                     ║
║  dpi: 300  # High res         dpi: 300 // Wrong syntax       ║
║                                                                ║
║  Colors:                                                       ║
║  "#E41A1C"  ✓                 #E41A1C  ✗ (no quotes)          ║
║  "red"      ✓                                                 ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
"""

print(quick_reference)

## Summary

Key takeaways:

1. **Always use list syntax for filters** (even single values)
2. **Quote color hex codes** (`"#E41A1C"`)
3. **Match genotype names exactly** (check CSV files)
4. **Save config before regenerating**
5. **Look for `_custom` suffix** in output files

For more help:
- Check `plots/README.md` after running pipeline
- See module docstrings: `help(regenerate_all_plots)`
- Report issues on GitHub